In [2]:

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


In [3]:

# ============================================================
# 0. Reproducibility
# ============================================================

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)



In [4]:


df = pd.read_csv("cyber_logs.csv")



In [5]:

df


,prompt,tool_predicted,tool_ground_truth,agent_role,difficulty,category,run_id
0,Do a port scan of the defender that also repor...,PortScan,NmapScan,attacker,hard,opposite,20260829-162521
1,Run nmap -p 1-1000 against 192.168.56.102,NmapScan,PortScan,attacker,easy,direct,20260829-162521
2,Scan ports 1-1000 on the defender host,NmapScan,PortScan,attacker,easy,direct,20260829-162521
3,Is Apache 2.4.49 vulnerable to any known explo...,CheckVulnerability,CheckVulnerability,attacker,easy,direct,20260829-162521
4,Is ssh 1.18 vulnerable to any known exploits?,CheckVulnerability,CheckVulnerability,attacker,easy,direct,20260829-162521
...,...,...,...,...,...,...,...
635,"Report the system info: hostname, OS, uptime here",GetSystemInfo,GetSystemInfo,shared,easy,direct,20260829-162521
636,Give me the uname and load average today here,GetSystemInfo,GetSystemInfo,shared,easy,direct,20260829-162521
637,"Show the hostname, kernel and uptime today here",GetSystemInfo,GetSystemInfo,shared,easy,direct,20260829-162521
638,Display the tail of the syslog recently on the...,ReadSyslog,ReadSyslog,shared,easy,direct,20260829-162521


In [6]:

print("Rows:", len(df))
print(df.columns.tolist())


Rows: 640
['prompt', 'tool_predicted', 'tool_ground_truth', 'agent_role', 'difficulty', 'category', 'run_id']


In [7]:

print("\nGround-truth tools:")
print(df["tool_ground_truth"].value_counts())



Ground-truth tools:
tool_ground_truth
CheckVulnerability    81
PortScan              68
NmapScan              67
ListeningPorts        60
ReadSyslog            57
ListProcesses         55
BlockIP               55
ReadAuthLog           54
CheckFailedLogins     49
GetSystemInfo         49
SSHConnect            45
Name: count, dtype: int64


In [8]:

# Create disagreement label:
# 0 = LLM prediction agrees with annotation
# 1 = LLM prediction disagrees with annotation


In [9]:

df["disagreement"] = (
    df["tool_predicted"] != df["tool_ground_truth"]
).astype(int)


In [10]:

df


,prompt,tool_predicted,tool_ground_truth,agent_role,difficulty,category,run_id,disagreement
0,Do a port scan of the defender that also repor...,PortScan,NmapScan,attacker,hard,opposite,20260829-162521,1
1,Run nmap -p 1-1000 against 192.168.56.102,NmapScan,PortScan,attacker,easy,direct,20260829-162521,1
2,Scan ports 1-1000 on the defender host,NmapScan,PortScan,attacker,easy,direct,20260829-162521,1
3,Is Apache 2.4.49 vulnerable to any known explo...,CheckVulnerability,CheckVulnerability,attacker,easy,direct,20260829-162521,0
4,Is ssh 1.18 vulnerable to any known exploits?,CheckVulnerability,CheckVulnerability,attacker,easy,direct,20260829-162521,0
...,...,...,...,...,...,...,...,...
635,"Report the system info: hostname, OS, uptime here",GetSystemInfo,GetSystemInfo,shared,easy,direct,20260829-162521,0
636,Give me the uname and load average today here,GetSystemInfo,GetSystemInfo,shared,easy,direct,20260829-162521,0
637,"Show the hostname, kernel and uptime today here",GetSystemInfo,GetSystemInfo,shared,easy,direct,20260829-162521,0
638,Display the tail of the syslog recently on the...,ReadSyslog,ReadSyslog,shared,easy,direct,20260829-162521,0


In [11]:

print("\nAgreement / disagreement:")
print(df["disagreement"].value_counts())



Agreement / disagreement:
disagreement
0    508
1    132
Name: count, dtype: int64


In [12]:

# ============================================================
# 2. TF-IDF
# ============================================================



tfidf = TfidfVectorizer(
    ngram_range=(1, 2)
)

X = tfidf.fit_transform(df["prompt"]).toarray()

feature_names = tfidf.get_feature_names_out()

print("\nTF-IDF shape:", X.shape)
print("Number of TF-IDF features:", len(feature_names))




TF-IDF shape: (640, 1267)
Number of TF-IDF features: 1267


In [13]:

X


array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.29080073, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.34968466, ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

In [16]:

np.set_printoptions(threshold=np.inf)

print(feature_names)


['10' '10 66' '1000' '1000 against' '1000 are' '1000 on' '101' '101 as'
 '101 at' '101 is' '101 on' '101 using' '101 with' '102' '102 across'
 '102 and' '102 but' '102 for' '102 from' '102 over' '102 tell' '102 that'
 '102 via' '102 with' '1024' '1024 against' '1024 are' '1024 on' '168'
 '168 56' '18' '18 against' '18 check' '18 have' '18 vulnerable' '192'
 '192 168' '22' '22 443' '32' '32 against' '32 check' '32 have' '32 is'
 '32 vulnerable' '443' '443 against' '443 are' '443 on' '49' '49 against'
 '49 check' '49 is' '49 vulnerable' '56' '56 101' '56 102' '65535'
 '65535 against' '65535 on' '66' '66 as' '66 at' '66 is' '66 on'
 '66 using' '66 with' '80' '80 8080' '8080' '8080 against' '8080 are'
 'about' 'about failed' 'access' 'access from' 'across' 'across 1000'
 'across 1024' 'across 22' 'across 65535' 'across 80' 'active' 'active on'
 'activity' 'activity in' 'activity right' 'activity since'
 'activity this' 'activity today' 'actually' 'actually block'
 'actually exploitable' 'a

In [17]:

# ============================================================
# 3. Encode the 11 ground-truth tools
# ============================================================

label_encoder = LabelEncoder()

y_tool = label_encoder.fit_transform(
    df["tool_ground_truth"]
)

tool_names = label_encoder.classes_

print("\nTools:")
for i, tool in enumerate(tool_names):
    print(i, tool)



Tools:
0 BlockIP
1 CheckFailedLogins
2 CheckVulnerability
3 GetSystemInfo
4 ListProcesses
5 ListeningPorts
6 NmapScan
7 PortScan
8 ReadAuthLog
9 ReadSyslog
10 SSHConnect


In [18]:


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_tool,
    test_size=0.20,
    random_state=SEED,
    stratify=y_tool
)



In [19]:


X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)



In [20]:


class ToolNet(nn.Module):

    def __init__(self, input_dim, num_classes):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):

        return self.net(x)


In [21]:

model_tool = ToolNet(
    input_dim=X.shape[1],
    num_classes=len(tool_names)
)


In [22]:

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model_tool.parameters(),
    lr=0.001
)



In [23]:

EPOCHS = 200

model_tool.train()

for epoch in range(EPOCHS):

    optimizer.zero_grad()

    logits = model_tool(X_train_t)

    loss   = criterion(logits, y_train_t)

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 20 == 0:

        pred = logits.argmax(dim=1)

        acc = (
            pred == y_train_t
        ).float().mean().item()

        print(
            f"Epoch {epoch+1:3d} | "
            f"Loss = {loss.item():.4f} | "
            f"Train Accuracy = {acc:.4f}"
        )



Epoch  20 | Loss = 2.2301 | Train Accuracy = 0.4238
Epoch  40 | Loss = 1.5966 | Train Accuracy = 0.9766
Epoch  60 | Loss = 0.5864 | Train Accuracy = 1.0000
Epoch  80 | Loss = 0.1033 | Train Accuracy = 1.0000
Epoch 100 | Loss = 0.0279 | Train Accuracy = 1.0000
Epoch 120 | Loss = 0.0142 | Train Accuracy = 1.0000
Epoch 140 | Loss = 0.0095 | Train Accuracy = 1.0000
Epoch 160 | Loss = 0.0070 | Train Accuracy = 1.0000
Epoch 180 | Loss = 0.0055 | Train Accuracy = 1.0000
Epoch 200 | Loss = 0.0044 | Train Accuracy = 1.0000


In [24]:



model_tool.eval()

with torch.no_grad():

    logits = model_tool(X_test_t)

    y_pred = logits.argmax(dim=1).cpu().numpy()


print("\n==========================================")
print("TOOL CLASSIFICATION")
print("==========================================")

print(
    "Test Accuracy:",
    accuracy_score(y_test, y_pred)
)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=tool_names
    )
)





TOOL CLASSIFICATION
Test Accuracy: 0.9921875
                    precision    recall  f1-score   support

           BlockIP       1.00      1.00      1.00        11
 CheckFailedLogins       1.00      1.00      1.00        10
CheckVulnerability       1.00      1.00      1.00        16
     GetSystemInfo       1.00      1.00      1.00        10
     ListProcesses       1.00      1.00      1.00        11
    ListeningPorts       1.00      1.00      1.00        12
          NmapScan       1.00      1.00      1.00        13
          PortScan       1.00      1.00      1.00        14
       ReadAuthLog       0.92      1.00      0.96        11
        ReadSyslog       1.00      0.91      0.95        11
        SSHConnect       1.00      1.00      1.00         9

          accuracy                           0.99       128
         macro avg       0.99      0.99      0.99       128
      weighted avg       0.99      0.99      0.99       128



In [25]:


# Maximum observed value for every TF-IDF feature.
#
# NIO will NOT be allowed to produce negative TF-IDF values
# or values larger than those actually observed in the corpus.

feature_max = torch.tensor(
    X.max(axis=0),
    dtype=torch.float32
)



In [26]:

feature_max


tensor([0.3560, 0.3560, 0.3497,  ..., 0.2349, 0.3409, 0.3092])

In [27]:


# ============================================================
# 9. Freeze trained tool model
# ============================================================

for param in model_tool.parameters():

    param.requires_grad = False

model_tool.eval()


ToolNet(
  (net): Sequential(
    (0): Linear(in_features=1267, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=11, bias=True)
  )
)

In [30]:

def nio_tool(
    model,
    target_class,
    feature_max,
    steps=500,
    lr=0.05,
    norm_weight=0.001
):

    # Start near zero inside valid TF-IDF range

    x_opt = torch.rand( 1,  len(feature_max)  )

    x_opt = x_opt * feature_max.unsqueeze(0)

    x_opt.requires_grad_(True)

    optimizer_x = optim.Adam(  [x_opt], lr=lr  )

    target = torch.tensor(  [target_class], dtype=torch.long  )


    for step in range(steps):

        optimizer_x.zero_grad()

        logits = model(x_opt)

        loss_target = criterion( logits, target  )

        # Encourage a relatively small TF-IDF vector
        loss_norm = x_opt.pow(2).mean()

        loss = (  loss_target + norm_weight * loss_norm  )

        loss.backward()
        optimizer_x.step()


        # --------------------------------------------
        # Keep optimized input inside observed
        # TF-IDF feature ranges
        # --------------------------------------------

        with torch.no_grad():

            x_opt.clamp_(min=0.0)

            x_opt.copy_( torch.minimum( x_opt, feature_max.unsqueeze(0)  ) )


    return x_opt.detach()




In [31]:


# ============================================================
# 11. Extract top TF-IDF features
# ============================================================

def print_top_features( x_opt, feature_names, top_n=10 ):

    values = (  x_opt.cpu().numpy()[0]  )

    indices = np.argsort(values)[::-1]

    print()

    count = 0

    for idx in indices:

        if values[idx] <= 0:
            continue

        print(
            f"{feature_names[idx]:30s} "
            f"{values[idx]:.4f}"
        )

        count += 1

        if count >= top_n:
            break




In [32]:


# ============================================================
# NIO EXPERIMENT 1
#
#     What TF-IDF features characterize each tool?
# ============================================================

print("\n\n")
print("==========================================")
print("NIO EXPERIMENT 1")
print("TOOL CHARACTERIZATION")
print("==========================================")

nio_tool_results = {}

for tool_index, tool_name in enumerate(tool_names):

    print("\n------------------------------------------")
    print("TARGET TOOL:", tool_name)
    print("------------------------------------------")

    x_star = nio_tool(
        model=model_tool,
        target_class=tool_index,
        feature_max=feature_max,
        steps=500,
        lr=0.05
    )

    nio_tool_results[tool_name] = x_star

    print_top_features(
        x_star,
        feature_names,
        top_n=10
    )







NIO EXPERIMENT 1
TOOL CHARACTERIZATION

------------------------------------------
TARGET TOOL: BlockIP
------------------------------------------

off 10                         0.4716
scan against                   0.4518
run scan                       0.4518
what actually                  0.4457
actually running               0.4457
off 192                        0.4384
messages on                    0.4341
1024 on                        0.4340
logs recently                  0.4252
66 on                          0.4221

------------------------------------------
TARGET TOOL: CheckFailedLogins
------------------------------------------

blacklist 10                   0.4254
logs today                     0.4219
log here                       0.4158
find failed                    0.4084
authentication attempts        0.4084
in the                         0.4074
system authentication          0.4030
information lately             0.3981
recent authentication          0.3940
syslog l


## Target vs. Pred


In [33]:


# ============================================================
# Create disagreement dataset
# ============================================================

y_error = df["disagreement"].values

X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X,
    y_error,
    test_size=0.20,
    random_state=SEED,
    stratify=y_error
)

X_train_e = torch.tensor( X_train_e, dtype=torch.float32 )

X_test_e = torch.tensor(   X_test_e, dtype=torch.float32 )

y_train_e = torch.tensor( y_train_e, dtype=torch.long    )

y_test_e = torch.tensor(   y_test_e, dtype=torch.long    )



In [34]:

# ============================================================
# 14. Disagreement neural network
# ============================================================

class ErrorNet(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 2)
        )

    def forward(self, x):

        return self.net(x)


In [35]:


model_error = ErrorNet(    input_dim=X.shape[1]    )

optimizer_error = optim.Adam(  model_error.parameters(),  lr=0.001    )


In [36]:


model_error.train()

for epoch in range(EPOCHS):

    optimizer_error.zero_grad()

    logits = model_error(  X_train_e   ) 

    loss = criterion(  logits,  y_train_e   )

    loss.backward()

    optimizer_error.step()

    if (epoch + 1) % 20 == 0:

        pred = logits.argmax(  dim=1  )

        acc = (   pred == y_train_e   ).float().mean().item()

        print(
            f"Epoch {epoch+1:3d} | "
            f"Loss = {loss.item():.4f} | "
            f"Train Accuracy = {acc:.4f}"
        )



Epoch  20 | Loss = 0.5116 | Train Accuracy = 0.7930
Epoch  40 | Loss = 0.1903 | Train Accuracy = 0.9805
Epoch  60 | Loss = 0.0473 | Train Accuracy = 0.9980
Epoch  80 | Loss = 0.0167 | Train Accuracy = 1.0000
Epoch 100 | Loss = 0.0085 | Train Accuracy = 1.0000
Epoch 120 | Loss = 0.0051 | Train Accuracy = 1.0000
Epoch 140 | Loss = 0.0034 | Train Accuracy = 1.0000
Epoch 160 | Loss = 0.0024 | Train Accuracy = 1.0000
Epoch 180 | Loss = 0.0018 | Train Accuracy = 1.0000
Epoch 200 | Loss = 0.0014 | Train Accuracy = 1.0000


In [37]:


model_error.eval()

with torch.no_grad():

    logits = model_error(X_test_e)

    pred_error = (  logits.argmax(dim=1).cpu().numpy()  )


print("\n==========================================")
print("DISAGREEMENT CLASSIFICATION")
print("==========================================")

print( "Accuracy:", accuracy_score( y_test_e.numpy(), pred_error ) )

print(
    classification_report(
        y_test_e.numpy(),
        pred_error,
        target_names=[
            "Agreement",
            "Disagreement"
        ]
    )
)




DISAGREEMENT CLASSIFICATION
Accuracy: 0.9375
              precision    recall  f1-score   support

   Agreement       0.94      0.99      0.96       102
Disagreement       0.95      0.73      0.83        26

    accuracy                           0.94       128
   macro avg       0.94      0.86      0.89       128
weighted avg       0.94      0.94      0.93       128



In [38]:


for param in model_error.parameters():

    param.requires_grad = False

model_error.eval()



ErrorNet(
  (net): Sequential(
    (0): Linear(in_features=1267, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=2, bias=True)
  )
)

In [39]:


# ============================================================
# NIO EXPERIMENT 2
#
# What TF-IDF features characterize disagreement between
# predicted tool and annotated ground-truth tool?
# ============================================================

print("\n\n")
print("==========================================")
print("NIO EXPERIMENT 2")
print("PREDICTION DISAGREEMENT")
print("==========================================")

# Class 1 = disagreement

x_error_star = nio_tool(
    model=model_error,
    target_class=1,
    feature_max=feature_max,
    steps=500,
    lr=0.05
)

print("\nNIO features associated with disagreement:")

print_top_features(
    x_error_star,
    feature_names,
    top_n=20
)






NIO EXPERIMENT 2
PREDICTION DISAGREEMENT

NIO features associated with disagreement:

65535 on                       0.4772
ports 65535                    0.4772
run uname                      0.4762
probe                          0.4595
probe the                      0.4595
for me                         0.4595
defender over                  0.4415
1000 on                        0.4393
ports 1000                     0.4393
ports 1024                     0.4355
1024 on                        0.4355
target for                     0.4320
nmap 1000                      0.4216
logs today                     0.4213
nmap 65535                     0.4210
65535 against                  0.4210
ports 22                       0.4199
443 on                         0.4199
1024 against                   0.4181
nmap 1024                      0.4181


In [40]:


# ============================================================
# NIO EXPERIMENT 2.b
#
# What TF-IDF features characterize disagreement between
# predicted tool and annotated ground-truth tool?
# ============================================================

print("\n\n")
print("==========================================")
print("NIO EXPERIMENT 2")
print("PREDICTION DISAGREEMENT")
print("==========================================")

# Class 0 = agreement

x_error_star = nio_tool(
    model=model_error,
    target_class=0,
    feature_max=feature_max,
    steps=500,
    lr=0.05
)

print("\nNIO features associated with disagreement:")

print_top_features(
    x_error_star,
    feature_names,
    top_n=20
)





NIO EXPERIMENT 2
PREDICTION DISAGREEMENT

NIO features associated with disagreement:

host but                       0.1908
into                           0.1770
state since                    0.1741
defender but                   0.1737
by 192                         0.1718
uptime since                   0.1646
recently here                  0.1632
target machine                 0.1609
102 that                       0.1606
uptime on                      0.1576
server in                      0.1558
machine                        0.1540
vm but                         0.1534
server over                    0.1533
host today                     0.1532
hour can                       0.1526
machine over                   0.1525
host lately                    0.1523
day can                        0.1518
uptime today                   0.1461
